# Motoneuron Three-State Temporal Screen

This notebook applies the temporal screen to the real motoneuron recordings **without pretending that directed ground truth is known**. It therefore does not report true-edge coverage or direction accuracy. Instead it measures:

- directional, ambiguous, and unmatched pair fractions;
- total candidate density after ambiguous pairs retain both directions;
- direction replication across held-out temporal blocks;
- episode-wise timing-shift null diagnostics.

A diagnostic pass only justifies a later unrestricted-versus-prior c-GC comparison. It is not evidence of causal connectivity. Nothing executes by default.

In [ ]:
from pathlib import Path
import csv
import json
import os
import shlex
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'calcium_transient_rising_flank').is_dir():
            return candidate
        nested = candidate / 'calcium-transient-rising-flank'
        if (nested / 'src' / 'calcium_transient_rising_flank').is_dir():
            return nested
    raise RuntimeError('Run from the repository, package root, or notebooks directory.')


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
def command_text(command: list[str]) -> str:
    return ' '.join(shlex.quote(str(part)) for part in command)


def run_or_print(command: list[str], *, execute: bool) -> None:
    print(command_text(command))
    if not execute:
        print('Dry run only. Set RUN_SCREEN = True when ready.')
        return
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
    env.setdefault('MPLCONFIGDIR', '/tmp/rising-flanks-matplotlib')
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)

## Prespecified empirical configuration

Case D is the initial screen because it removes bad neurons and artefacts and uses the smoothed traces. Fixed temporal windows are the primary cross-fitting strata; unlike population-defined bouts, they do not use the same rising-flank activity to define both episodes and directions. At 4 Hz, `WINDOW_FRAMES = 240` corresponds to one minute. Rerun 120 and 480 frames as a segmentation sensitivity analysis before interpreting a pass.

In [ ]:
RUN_SCREEN = False

DATA_DIR = PROJECT_ROOT / 'data' / 'motoneurons'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'motorneurons' / 'temporal_screen_manual'
CASES = 'D'
RECORDINGS = 'all'  # or e.g. 'F3T1,F3T2,F5T2'
MAX_ONSET_LAGS = '1,2,3'
DEADBANDS = '0,1'
TOLERANCE = 0.0
MIN_RUN_SAMPLES = 2
SEGMENT_MODE = 'fixed_windows'
WINDOW_FRAMES = 240
MERGE_GAP_FRAMES = 3  # used only by population_bouts sensitivity
MINIMUM_PARTICIPATING_ROIS = 2
N_NULLS = 20
RANDOM_STATE = 20260821

In [ ]:
command = [
    PYTHON,
    'examples/motorneuron_temporal_screen.py',
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--cases', CASES,
    '--recordings', RECORDINGS,
    '--max-onset-lags', MAX_ONSET_LAGS,
    '--deadbands', DEADBANDS,
    '--tolerance', str(TOLERANCE),
    '--min-run-samples', str(MIN_RUN_SAMPLES),
    '--segment-mode', SEGMENT_MODE,
    '--window-frames', str(WINDOW_FRAMES),
    '--merge-gap-frames', str(MERGE_GAP_FRAMES),
    '--minimum-participating-rois', str(MINIMUM_PARTICIPATING_ROIS),
    '--n-nulls', str(N_NULLS),
    '--random-state', str(RANDOM_STATE),
]
run_or_print(command, execute=RUN_SCREEN)

## Inspect your truth-free screen

The table below is descriptive. `passes_empirical_diagnostics` requires candidate density ≤ 0.60, held-out directional replication ≥ 0.80, and observed directional fraction above the 95th percentile of the timing-shift nulls. It does not replace the unavailable ground-truth accuracy and coverage gates.

In [ ]:
summary_path = OUTPUT_DIR / 'screen_summary.csv'
if summary_path.is_file():
    with summary_path.open(newline='', encoding='utf-8') as handle:
        summary_rows = list(csv.DictReader(handle))
    columns = [
        'case', 'recording', 'max_onset_lag_frames', 'deadband_frames',
        'candidate_density_mean', 'heldout_directional_replication_mean',
        'directional_fraction_above_null_p95_mean',
        'passes_empirical_diagnostics',
    ]
    for row in summary_rows:
        print({column: row[column] for column in columns})
else:
    summary_rows = []
    print(f'No manual empirical result yet at {summary_path}')

In [ ]:
if summary_rows:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6.4, 4.2), constrained_layout=True)
    for deadband, marker in [('0', 'o'), ('1', 's')]:
        selected = [row for row in summary_rows if row['deadband_frames'] == deadband]
        density = [float(row['candidate_density_mean']) for row in selected]
        replication = [
            float(row['heldout_directional_replication_mean'])
            for row in selected
            if row['heldout_directional_replication_mean']
        ]
        density = [
            float(row['candidate_density_mean'])
            for row in selected
            if row['heldout_directional_replication_mean']
        ]
        ax.scatter(density, replication, marker=marker, label=f'deadband {deadband}', alpha=0.8)
    ax.axvline(0.60, color='0.35', linestyle='--', linewidth=1.0)
    ax.axhline(0.80, color='0.35', linestyle=':', linewidth=1.0)
    ax.set(xlabel='Three-state candidate density', ylabel='Held-out direction replication', xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)
    plt.show()

## Interpretation checklist

Before any motorneuron c-GC follow-up:

1. require the diagnostic to replicate across recordings, not merely settings within one recording;
2. rerun at 120, 240, and 480-frame validation windows;
3. treat deadband 0 as primary and deadband 1 as sensitivity;
4. compare unrestricted, three-state, and soft-prior c-GC on the same held-out blocks;
5. retain the existing temporal null and graph-stability controls;
6. describe any inferred edge as a candidate directed association, not identified causality.